# Cost & Deployability Analysis

Quantitative cost model for the two-stage Gemini EHS detection pipeline.
All numbers derive from actual Vertex AI token pricing and measured token
counts from `run_logging/experiment_logger.py`.

**Sponsor context:** False positives are acceptable — the system is only
consulted when a real accident is confirmed (staff check whether a report
already exists for that timeframe). This shifts the key metric from
*cost per clip* to **cost per true positive report**.

In [ ]:
import os, sys
from pathlib import Path

if "__vsc_ipynb_file__" in dir():
    repo_root = str(Path(__vsc_ipynb_file__).parent.parent)
else:
    def _find_root():
        for p in [Path.cwd()] + list(Path.cwd().parents):
            if (p / "pipeline").is_dir():
                return str(p)
        raise RuntimeError("Could not find repo root.")
    repo_root = _find_root()

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option('display.float_format', '{:.6f}'.format)
print("cwd:", os.getcwd())

---
## 1 — Token Cost Model

Token counts and pricing come directly from `run_logging/experiment_logger.py`.
These are consistent across runs — relative comparisons are reliable even if
absolute counts are approximations.

In [ ]:
from run_logging.experiment_logger import (
    COST_PER_1K_INPUT_TOKENS,
    COST_PER_1K_OUTPUT_TOKENS,
    _STAGE1_INPUT_TOKENS,
    _STAGE1_OUTPUT_TOKENS,
    _STAGE2_INPUT_TOKENS,
    _STAGE2_OUTPUT_TOKENS,
)

# ── Per-call cost for each model ──────────────────────────────────────────────
MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite", "gemini-2.5-pro"]

def stage1_cost_per_call(model: str) -> float:
    """Cost of one Stage 1 binary gate call."""
    return (
        _STAGE1_INPUT_TOKENS  / 1000 * COST_PER_1K_INPUT_TOKENS[model] +
        _STAGE1_OUTPUT_TOKENS / 1000 * COST_PER_1K_OUTPUT_TOKENS[model]
    )

def stage2_cost_per_call(model: str) -> float:
    """Cost of one Stage 2 classification + EHS report call."""
    return (
        _STAGE2_INPUT_TOKENS  / 1000 * COST_PER_1K_INPUT_TOKENS[model] +
        _STAGE2_OUTPUT_TOKENS / 1000 * COST_PER_1K_OUTPUT_TOKENS[model]
    )

def per_clip_cost(
    s1_model: str = "gemini-2.5-flash",
    s2_model: str = "gemini-2.5-flash",
    n_votes: int = 3,
    accident_rate: float = 0.01,
) -> dict:
    """Expected cost per clip given an accident rate."""
    s1 = n_votes * stage1_cost_per_call(s1_model)
    s2 = stage2_cost_per_call(s2_model)
    expected = s1 + accident_rate * s2  # E[cost] = stage1 always + stage2 p(accident)
    return {
        "s1_cost":         round(s1, 8),
        "s2_cost":         round(s2, 8),
        "non_accident":    round(s1, 8),
        "accident_clip":   round(s1 + s2, 8),
        "expected_per_clip": round(expected, 8),
    }

# ── Print token pricing breakdown ─────────────────────────────────────────────
rows = []
for m in MODELS:
    rows.append({
        "Model":           m,
        "S1 input $/1K":  COST_PER_1K_INPUT_TOKENS[m],
        "S1 output $/1K": COST_PER_1K_OUTPUT_TOKENS[m],
        "S1 cost/call":   round(stage1_cost_per_call(m), 8),
        "S2 cost/call":   round(stage2_cost_per_call(m), 8),
        "S1×3 votes":     round(3 * stage1_cost_per_call(m), 8),
    })

df_pricing = pd.DataFrame(rows).set_index("Model")
print(f"Token counts: Stage 1 = {_STAGE1_INPUT_TOKENS} in / {_STAGE1_OUTPUT_TOKENS} out tokens")
print(f"              Stage 2 = {_STAGE2_INPUT_TOKENS} in / {_STAGE2_OUTPUT_TOKENS} out tokens\n")
print(df_pricing.to_string())

---
## 2 — Accident Rate Sensitivity

How does cost scale with the actual accident rate in the footage?
Stage 1 is always paid; Stage 2 is only paid when Stage 1 fires.

In [ ]:
# Best config from ablation: flash/flash, n_votes=3
# binary_precision=0.952 → 95.2% of Stage 2 calls are true positives
# binary_recall=0.930    → 7% of real accidents are missed

S1_MODEL       = "gemini-2.5-flash"
S2_MODEL       = "gemini-2.5-flash"
N_VOTES        = 3
BINARY_PREC    = 0.952   # precision of Stage 1 gate
BINARY_RECALL  = 0.930   # recall of Stage 1 gate

S1_PER_CLIP    = N_VOTES * stage1_cost_per_call(S1_MODEL)
S2_PER_CALL    = stage2_cost_per_call(S2_MODEL)

ACCIDENT_RATES = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]

rows = []
for ar in ACCIDENT_RATES:
    # Expected Stage 1 positives per 1000 clips
    #   True accidents:  1000 * ar
    #   Stage 2 calls:   TPs + FPs
    #   precision = TP/(TP+FP)  →  FP = TP * (1/prec - 1)
    n_clips     = 1000
    n_accidents = n_clips * ar
    n_tp        = n_accidents * BINARY_RECALL          # correctly detected
    n_fp        = n_tp * (1 / BINARY_PREC - 1)        # false positives reaching Stage 2
    n_fn        = n_accidents * (1 - BINARY_RECALL)   # missed accidents
    n_s2_calls  = n_tp + n_fp                         # total Stage 2 invocations

    s1_total    = n_clips * S1_PER_CLIP
    s2_total    = n_s2_calls * S2_PER_CALL
    total_cost  = s1_total + s2_total

    cost_per_tp = (total_cost / n_tp) if n_tp > 0 else float("inf")
    fp_overhead = (n_fp / n_s2_calls * 100) if n_s2_calls > 0 else 0

    rows.append({
        "Accident rate":       f"{ar:.1%}",
        "Accidents / 1K clips": round(n_accidents, 1),
        "Stage 2 calls / 1K":  round(n_s2_calls, 1),
        "FP Stage 2 calls":    round(n_fp, 2),
        "FP overhead %":        round(fp_overhead, 1),
        "Total cost / 1K clips": round(total_cost, 4),
        "Cost per TP report ($)": round(cost_per_tp, 4),
        "Missed accidents (FN)": round(n_fn, 2),
    })

df_sens = pd.DataFrame(rows).set_index("Accident rate")
pd.set_option('display.float_format', '{:.4f}'.format)
print("Sensitivity Table — Flash/Flash (n_votes=3), per 1,000 clips\n")
print(df_sens.to_string())

---
## 3 — Sponsor Context: FP Economics

The sponsor confirmed false positives are acceptable because the system is
only accessed when a real accident is confirmed — staff check whether a report
was auto-generated in that timeframe. An unused EHS report costs Stage 2 API
fees but causes no operational harm. False **negatives** are the real loss:
no report is generated when an accident occurred.

In [ ]:
# At the realistic 1% accident rate
AR = 0.01
N  = 1000

n_accidents = N * AR                              # 10
n_tp        = n_accidents * BINARY_RECALL         # ~9.3
n_fp        = n_tp * (1 / BINARY_PREC - 1)       # ~0.47
n_fn        = n_accidents - n_tp                 # ~0.7 missed
n_s2        = n_tp + n_fp

fp_pct      = n_fp / n_s2 * 100
useful_pct  = n_tp / n_s2 * 100
total_cost  = N * S1_PER_CLIP + n_s2 * S2_PER_CALL
cost_per_tp = total_cost / n_tp

print(f"At {AR:.0%} accident rate, per 1,000 clips:")
print(f"  Real accidents:               {n_accidents:.0f}")
print(f"  Detected (TP):                {n_tp:.1f}  → EHS report generated")
print(f"  False positives (FP):         {n_fp:.2f} → EHS report generated but unused")
print(f"  Missed (FN):                  {n_fn:.2f}  → no report generated  ← only real loss")
print()
print(f"  Stage 2 calls:                {n_s2:.1f}")
print(f"    Useful (TP):                {useful_pct:.1f}% of Stage 2 spend produces a real report")
print(f"    Wasted (FP):                {fp_pct:.1f}% of Stage 2 spend is overhead")
print()
print(f"  Total cost / 1,000 clips:     ${total_cost:.4f}")
print(f"  Cost per true positive report: ${cost_per_tp:.4f}")
print(f"  Stage 1 share of total cost:  {N * S1_PER_CLIP / total_cost:.1%}")
print(f"  Stage 2 share of total cost:  {n_s2 * S2_PER_CALL / total_cost:.1%}")

---
## 4 — Multi-Camera Daily Cost

Models a surveillance deployment where each camera produces one clip every
30 seconds (120 clips/hour/camera, 2,880 clips/day/camera).

In [ ]:
CLIPS_PER_HOUR  = 120   # one 30-second clip per camera per 30 seconds
CLIPS_PER_DAY   = CLIPS_PER_HOUR * 24   # 2,880 clips/camera/day

CAMERA_COUNTS = [1, 5, 10, 25, 50, 100, 500]
ACCIDENT_RATE = 0.01

# Compare flash/flash (best) vs flash-lite/flash (cheapest viable)
configs = {
    "Flash/Flash (best, n=3)": {
        "s1": "gemini-2.5-flash", "s2": "gemini-2.5-flash", "n": 3,
    },
    "Flash-Lite/Flash (cheapest S1, n=3)": {
        "s1": "gemini-2.5-flash-lite", "s2": "gemini-2.5-flash", "n": 3,
    },
}

for cfg_name, cfg in configs.items():
    s1c = cfg["n"] * stage1_cost_per_call(cfg["s1"])
    s2c = stage2_cost_per_call(cfg["s2"])

    # Expected cost per clip
    n_tp_frac = ACCIDENT_RATE * BINARY_RECALL
    n_fp_frac = n_tp_frac * (1 / BINARY_PREC - 1)
    s2_rate   = n_tp_frac + n_fp_frac           # fraction of clips that trigger S2
    expected  = s1c + s2_rate * s2c

    rows = []
    for n_cams in CAMERA_COUNTS:
        daily_clips = n_cams * CLIPS_PER_DAY
        daily_cost  = daily_clips * expected
        monthly     = daily_cost * 30
        tp_per_day  = daily_clips * n_tp_frac
        cost_per_tp = daily_cost / tp_per_day if tp_per_day > 0 else float("inf")
        rows.append({
            "Cameras":         n_cams,
            "Clips/day":       f"{daily_clips:,}",
            "Daily cost ($)":  f"{daily_cost:.4f}",
            "Monthly ($)":     f"{monthly:.2f}",
            "TP reports/day":  f"{tp_per_day:.1f}",
            "$/TP report":     f"{cost_per_tp:.4f}",
        })

    print(f"\n{'─' * 70}")
    print(f"  {cfg_name}")
    print(f"  S1={cfg['s1']}  S2={cfg['s2']}  n_votes={cfg['n']}")
    print(f"  Expected cost/clip = ${expected:.8f}  (at {ACCIDENT_RATE:.0%} accident rate)")
    print(f"{'─' * 70}")
    print(pd.DataFrame(rows).set_index("Cameras").to_string())

---
## 5 — Breakeven vs Human Monitoring

In [ ]:
# Human reviewer cost
HUMAN_HOURLY_USD  = 25.0      # USD/hr for a dedicated safety monitor
HOURS_PER_DAY     = 24
HUMAN_DAILY_USD   = HUMAN_HOURLY_USD * HOURS_PER_DAY   # one reviewer covering 24/7

# Pipeline daily cost per camera (Flash/Flash, 1% accident rate)
s1c_ff = 3 * stage1_cost_per_call("gemini-2.5-flash")
s2c_ff = stage2_cost_per_call("gemini-2.5-flash")
n_tp_f = 0.01 * BINARY_RECALL
n_fp_f = n_tp_f * (1 / BINARY_PREC - 1)
expected_ff  = s1c_ff + (n_tp_f + n_fp_f) * s2c_ff
daily_per_cam = expected_ff * CLIPS_PER_DAY

breakeven_cameras = HUMAN_DAILY_USD / daily_per_cam

print(f"Human reviewer cost:   ${HUMAN_HOURLY_USD:.2f}/hr  →  ${HUMAN_DAILY_USD:.2f}/day (24/7)")
print(f"Pipeline cost/camera/day: ${daily_per_cam:.4f}")
print(f"Breakeven cameras:     {breakeven_cameras:.0f} cameras per human reviewer")
print()
print(f"  A single 24/7 human reviewer could be replaced by the pipeline")
print(f"  monitoring ~{breakeven_cameras:.0f} cameras at equivalent annual cost.")
print()

# Concrete data centre example
print("Concrete example — 50-camera data centre:")
daily_50 = 50 * daily_per_cam
annual_50 = daily_50 * 365
human_annual = HUMAN_DAILY_USD * 365
print(f"  Pipeline daily cost:  ${daily_50:.2f}")
print(f"  Pipeline annual cost: ${annual_50:,.2f}")
print(f"  Human annual cost:    ${human_annual:,.2f}")
print(f"  Savings:              ${human_annual - annual_50:,.2f}/year  ({(human_annual - annual_50) / human_annual:.0%} reduction)")

---
## 6 — Cost vs Quality Tradeoff Chart

In [ ]:
# Use actual ablation results for cost vs F1 Pareto comparison
# (flash/flash best, flash-lite/flash cheapest viable, flash/pro most expensive)

configs_plot = [
    # (label, binary_f1, macro_f1, cost_per_1k_clips_at_1pct)
    ("Flash/Flash\n(A_026, best)",        0.941, 0.635, None),
    ("Flash/Flash\n(G_005, best Phase 3)", 0.933, 0.671, None),
    ("Flash/Flash\n(H_000, CoT)",          0.916, 0.678, None),
    ("Flash-Lite/Flash\n(H_002)",          0.805, 0.577, None),
    ("Flash/Pro\n(H_001)",                 0.892, 0.628, None),
]

s1_models = {
    "A_026": "gemini-2.5-flash",
    "G_005": "gemini-2.5-flash",
    "H_000": "gemini-2.5-flash",
    "H_002": "gemini-2.5-flash-lite",
    "H_001": "gemini-2.5-flash",
}
s2_models = {
    "A_026": "gemini-2.5-flash",
    "G_005": "gemini-2.5-flash",
    "H_000": "gemini-2.5-flash",
    "H_002": "gemini-2.5-flash",
    "H_001": "gemini-2.5-pro",
}

keys  = ["A_026", "G_005", "H_000", "H_002", "H_001"]
bf1s  = [0.941, 0.933, 0.916, 0.805, 0.892]
mf1s  = [0.635, 0.671, 0.678, 0.577, 0.628]
labels = [
    "Flash/Flash\n(Phase 1 best)",
    "Flash/Flash\n(Phase 3 G_005)",
    "Flash/Flash\n(Phase 3 H_000)",
    "Lite/Flash\n(H_002)",
    "Flash/Pro\n(H_001)",
]

# Compute daily cost per 100 cameras
def daily_cost_100_cams(s1m, s2m, n=3, ar=0.01):
    s1c  = n * stage1_cost_per_call(s1m)
    s2c  = stage2_cost_per_call(s2m)
    n_tpf = ar * BINARY_RECALL
    n_fpf = n_tpf * (1 / BINARY_PREC - 1)
    exp   = s1c + (n_tpf + n_fpf) * s2c
    return 100 * CLIPS_PER_DAY * exp

costs = [daily_cost_100_cams(s1_models[k], s2_models[k]) for k in keys]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#2ecc71", "#3498db", "#1abc9c", "#e67e22", "#e74c3c"]

for ax, metric, title in [
    (axes[0], bf1s, "Binary F1"),
    (axes[1], mf1s, "Macro F1"),
]:
    sc = ax.scatter(costs, metric, c=colors, s=160, zorder=4, edgecolors="white", linewidths=1.5)
    for i, (x, y, lbl) in enumerate(zip(costs, metric, labels)):
        ax.annotate(lbl, (x, y), textcoords="offset points",
                    xytext=(8, -4), fontsize=8.5, color=colors[i])
    ax.set_xlabel("Daily cost, 100 cameras @ 1% accident rate ($)", fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f"Cost vs {title}", fontweight="bold", fontsize=12)
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))

fig.suptitle("Pipeline Configuration Tradeoff — Cost vs Classification Quality",
             fontsize=13, fontweight="bold")
plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/cost_quality_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/cost_quality_tradeoff.png")

---
## 7 — Throughput & Concurrency Model

Load test results from `api/load_test.py`. Run the load test first, then
reload this cell to visualise throughput vs concurrency.

In [ ]:
import json

load_test_path = "outputs/load_test_results.json"

if not os.path.exists(load_test_path):
    print(f"Load test results not found at {load_test_path}.")
    print("Run the load test first:")
    print("  # Terminal 1: uvicorn api.endpoint:app --host 0.0.0.0 --port 8000")
    print("  # Terminal 2: python api/load_test.py")
else:
    with open(load_test_path) as f:
        lt = json.load(f)

    results = lt["results"]
    conc    = [r["concurrency"]     for r in results]
    rps     = [r["throughput_rps"]  for r in results]
    p50     = [r["p50_ms"]          for r in results]
    p95     = [r["p95_ms"]          for r in results]
    p99     = [r["p99_ms"]          for r in results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(conc, rps, "o-", color="#2ecc71", lw=2, ms=8)
    for x, y in zip(conc, rps):
        ax1.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(4, 6), fontsize=9)
    ax1.set_xlabel("Concurrency (simultaneous requests)", fontsize=11)
    ax1.set_ylabel("Throughput (req/s)", fontsize=11)
    ax1.set_title("Throughput vs Concurrency", fontweight="bold", fontsize=12)
    ax1.grid(True, alpha=0.25)
    ax1.set_xticks(conc)

    ax2.plot(conc, p50, "o-", label="P50", color="#3498db", lw=2, ms=7)
    ax2.plot(conc, p95, "s--", label="P95", color="#e67e22", lw=2, ms=7)
    ax2.plot(conc, p99, "^:",  label="P99", color="#e74c3c", lw=2, ms=7)
    ax2.set_xlabel("Concurrency", fontsize=11)
    ax2.set_ylabel("Latency (ms)", fontsize=11)
    ax2.set_title("Latency Percentiles vs Concurrency", fontweight="bold", fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.25)
    ax2.set_xticks(conc)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}s"))

    fig.suptitle(f"Endpoint Load Test — {lt.get('clip', 'test clip')}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("outputs/load_test_chart.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Camera coverage
    best = max(results, key=lambda r: r["throughput_rps"])
    print(f"\nPeak: {best['throughput_rps']:.4f} req/s at concurrency={best['concurrency']}")
    for interval in [30, 60]:
        cams = best["throughput_rps"] * interval
        print(f"  → ~{cams:.0f} cameras in real time at {interval}s clip interval")
    print("Saved → outputs/load_test_chart.png")